In [11]:
import sys
import os
import numpy as np
import xarray as xr
import pandas as pd
import datetime
import fnmatch

import matplotlib as mpl
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [12]:
folder = '/Volumes/New_5TB/ESA_F4R/tropomi2/'
folder_out = '/Volumes/blue_wd/ESA_F4R/tropomi_merged_f1/' 
Y = 2020

In [13]:
#H2O column conversion

#molec/cm**2

#cm**2 --> kg
#(pressure in Pa)*( area in m**2)/(gravity in m/s**2)
P = 101300 #should this be 101325? 
A = 10**(-4)
g = 9.81 

conv_denom = P*A/g

#molec --> g
#(number of molecules)*(atomic weight in g/mol)/(avogadro number in molec/mol)
AW = 18.015
AV = 6.02214076*10**23


In [14]:
xls = []
lon_bnds, lat_bnds = (8, 32), (12,-15)
arr = fnmatch.filter(os.listdir(folder), '*'+str(Y)+'*')
print(arr)
for I,F in enumerate(arr):
    try:
        x = xr.open_dataset(folder+F,
                            group="PRODUCT",
                            drop_variables=['level','corner','glintflag','ground_pixel','nwin',
                                            'hdo_column_precision','hdo_column_apriori',
                                            'hdo_profile_apriori','h2o_profile_apriori',
                                            'delta_time','layer',
                                            'h2o_column_precision','h2o_column_apriori',
                                            'deltad_precision','hdo_column','h2o_vmr'])
        x = x.rename({'latitude':'lat','longitude':'lon'})
        x = x.assign_coords({"ground_pixel": x.ground_pixel.values})
        x = x.where((x.lat < 12.) & (x.lat > -15.),drop=True)
        x = x.where((x.lon < 31.) & (x.lon > 8.),drop=True)
        x = x.where(x['qa_value']>=1.0,drop=True)
        x['deltad'] = x['deltad']*1000.0
        N = x['h2o_column']
        conv_enum = N*AW/AV
        x['h2o_vmr'] = conv_enum/conv_denom
#        print(x['time.year'].max())
        if x['time.year'].max().values==Y:
            print('appending!')
            xls.append(x)
        x.close()
        print(F)
    except:
        print('not nc4: ',F)

['S5P_PAL__L2__HDO__S_20180831T102033_20180831T120203_04569_01_100300_20250307T221009.nc', 'S5P_PAL__L2__HDO__S_20180831T120203_20180831T134332_04570_01_100300_20250307T220803.nc', 'S5P_PAL__L2__HDO__S_20190412T102032_20190412T120202_07747_01_100300_20250308T062537.nc', 'S5P_PAL__L2__HDO__S_20190412T120202_20190412T134332_07748_01_100300_20250308T062854.nc', 'S5P_PAL__L2__HDO__S_20190602T110829_20190602T124959_08471_01_100300_20250308T020203.nc', 'S5P_PAL__L2__HDO__S_20190729T113850_20190729T132020_09280_01_100300_20250309T134418.nc', 'S5P_PAL__L2__HDO__S_20190729T132020_20190729T150149_09281_01_100300_20250309T134346.nc', 'S5P_PAL__L2__HDO__S_20200101T092016_20200101T110146_11492_01_100300_20250311T061441.nc', 'S5P_PAL__L2__HDO__S_20200101T110146_20200101T124316_11493_01_100300_20250311T060743.nc', 'S5P_PAL__L2__HDO__S_20200101T124316_20200101T142446_11494_01_100300_20250311T061107.nc', 'S5P_PAL__L2__HDO__S_20200102T090116_20200102T104246_11506_01_100300_20250311T013116.nc', 'S5P_PAL_

In [15]:
trp = xr.concat(xls,dim='time')
trp = trp.sortby('time')
#print(trp)
trp = trp.sel(time=slice(str(Y)+'-01-01',str(Y)+'-12-31'))
trp = trp.drop_vars(['qa_value','h2o_column'])
print(trp)

trp.to_netcdf(folder_out+"TROPOMI_merged_"+str(Y)+".nc",engine='h5netcdf')



<xarray.Dataset> Size: 4GB
Dimensions:       (time: 655, scanline: 1656, ground_pixel: 215)
Coordinates:
  * time          (time) datetime64[ns] 5kB 2020-01-01T11:23:20 ... 2020-12-3...
  * scanline      (scanline) int32 7kB 1195 1196 1197 1198 ... 2849 2850 2851
  * ground_pixel  (ground_pixel) int64 2kB 0 1 2 3 4 5 ... 210 211 212 213 214
    lat           (time, scanline, ground_pixel) float32 933MB nan nan ... nan
    lon           (time, scanline, ground_pixel) float32 933MB nan nan ... nan
Data variables:
    deltad        (time, scanline, ground_pixel) float32 933MB nan nan ... nan
    h2o_vmr       (time, scanline, ground_pixel) float32 933MB nan nan ... nan


FileNotFoundError: [Errno 2] Unable to synchronously create file (unable to open file: name = '/Volumes/blue_wd/ESA_F4R/tropomi_merged_f1/TROPOMI_merged_2020.nc', errno = 2, error message = 'No such file or directory', flags = 13, o_flags = 602)

In [ ]:
sys.exit()

In [ ]:
print(trp.lat.max().values)
print(trp.lat.min().values)

print(trp.lon.max().values)
print(trp.lon.min().values)

In [ ]:
#Plot climatology

tpN = trp.where((trp.lat < 12.) & (trp.lat > 5.),drop=True)
tpN = tpN.mean(dim=('scanline','ground_pixel'))
tpN = tpN.groupby('time.dayofyear').mean('time')
#print(tpN['deltad'])
#tpN['deltad'] = tpN['deltad']*1000.0
tpN['deltad'].plot()
plt.title('North TROPOMI')
#plt.ylim(0.2,0.85)
plt.show()
plt.clf()

tpE = trp.where((trp.lat < 5.) & (trp.lat > -5.),drop=True)
tpE = tpE.mean(dim=('scanline','ground_pixel'))
tpE = tpE.groupby('time.dayofyear').mean('time')
#tpE['deltad'] = tpE['deltad']*1000.0
tpE['deltad'].plot()
plt.title('Equatorial TROPOMI')
#plt.ylim(0.2,0.85)
plt.show()
plt.clf()

tpS = trp.where((trp.lat < -5.) & (trp.lat > -15.),drop=True)
tpS = tpS.mean(dim=('scanline','ground_pixel'))
tpS = tpS.groupby('time.dayofyear').mean('time')
tpS['deltad'].plot()
plt.title('South TROPOMI')
#plt.ylim(0.2,0.85)
plt.show()
plt.clf()